# MVP1 de SSD - Classificação do Posicionamento de Preço de Veículos

**Disciplina:** Sistemas de Suporte à Decisão  
**Universidade de Brasília (UnB)**  
**Aluno:** Juliano Teles Abrahao - 231013411

Este notebook apresenta um MVP de classificação do posicionamento de preço de veículos, utilizando uma base de dados com informações como **montadora, modelo, cor, grupo, situação, ano do modelo e Valor FIPE**.

**Objetivo:** treinar e comparar modelos de Machine Learning para classificar o posicionamento de preço dos veículos em 3 categorias:

- **Abaixo do esperado**
- **Dentro do esperado**
- **Acima do esperado**

A classificação é realizada comparando o **Valor FIPE de cada veículo com um valor de referência de veículos semelhantes**, permitindo identificar automaticamente se seu preço está abaixo, dentro ou acima do padrão esperado.

---

## Definição do Problema

O problema é tratado como uma **classificação multiclasse**.

Para cada veículo, é calculado um preço de referência com base na **mediana do Valor FIPE de veículos da mesma Montadora e Ano Modelo**.

A partir disso:

- **0 — Abaixo do esperado:** Valor FIPE < 90% do preço de referência
- **1 — Dentro do esperado:** Valor FIPE entre 90% e 110% do preço de referência
- **2 — Acima do esperado:** Valor FIPE > 110% do preço de referência

### Hipótese

Características como **Modelo, Grupo, Situação, Cor, Montadora e Ano Modelo** contêm informação suficiente para explicar se um veículo tende a ficar abaixo, dentro ou acima do padrão de preço do seu grupo de comparação.

> O Valor FIPE é usado apenas para construir a variável alvo e não entra como feature do modelo, evitando vazamento direto de informação.


## 1. Configuração do Ambiente


In [ ]:
# Execute esta célula no Google Colab.
# Se necessário:
# !pip install -q openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
print("Ambiente configurado com sucesso.")


## 2. Carga e Preparação dos Dados

A base esperada deve conter, no mínimo, as colunas:

- `Montadora`
- `Modelo`
- `Cor`
- `Grupo`
- `Situação`
- `Ano Modelo`
- `Valor FIPE`

O notebook aceita arquivos `.xlsx`, `.xls` ou `.csv`.


In [ ]:
print("Faça upload da base de veículos:")
uploaded = files.upload()
arquivo = next(iter(uploaded.keys()))

if arquivo.lower().endswith((".xlsx", ".xls")):
    df_raw = pd.read_excel(arquivo)
elif arquivo.lower().endswith(".csv"):
    try:
        df_raw = pd.read_csv(arquivo)
    except:
        df_raw = pd.read_csv(arquivo, sep=";")
else:
    raise ValueError("Formato não suportado. Utilize Excel ou CSV.")

print(f"Base carregada: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
display(df_raw.head())


### 2.1. Validação e limpeza


In [ ]:
colunas_necessarias = [
    "Montadora", "Modelo", "Cor", "Grupo",
    "Situação", "Ano Modelo", "Valor FIPE"
]

faltantes = [c for c in colunas_necessarias if c not in df_raw.columns]
if faltantes:
    raise ValueError(f"Colunas ausentes na base: {faltantes}")

df = df_raw.copy()

# Conversões
df["Ano Modelo"] = pd.to_numeric(df["Ano Modelo"], errors="coerce")
df["Valor FIPE"] = pd.to_numeric(df["Valor FIPE"], errors="coerce")

# Regras equivalentes às utilizadas no projeto original
df = df[df["Ano Modelo"] >= 1990]
df = df[df["Valor FIPE"] > 0]

# Remover nulos nas variáveis essenciais
df = df.dropna(subset=colunas_necessarias).copy()

print(f"Dataset após limpeza: {df.shape}")
display(df.head())


## 3. Engenharia da Variável Alvo

O posicionamento não será comparado com a mediana geral da base.

Para tornar a análise mais coerente, cada veículo é comparado inicialmente com veículos da mesma:

**Montadora + Ano Modelo**

Quando um grupo possui poucas observações, utilizamos uma regra de fallback:

1. Mediana de `Montadora + Ano Modelo`, quando há pelo menos 3 veículos;
2. Caso contrário, mediana da `Montadora`;
3. Caso ainda não seja possível, mediana geral da base.

Essa lógica permite criar um benchmark de preço adaptado ao contexto de cada veículo.


In [ ]:
# Estatísticas por Montadora + Ano Modelo
grupo_ref = (
    df.groupby(["Montadora", "Ano Modelo"])["Valor FIPE"]
      .agg(preco_referencia_grupo="median", n_grupo="size")
      .reset_index()
)

df = df.merge(grupo_ref, on=["Montadora", "Ano Modelo"], how="left")

# Fallback por montadora
mediana_montadora = df.groupby("Montadora")["Valor FIPE"].transform("median")
mediana_geral = df["Valor FIPE"].median()

df["preco_referencia"] = np.where(
    df["n_grupo"] >= 3,
    df["preco_referencia_grupo"],
    mediana_montadora
)

df["preco_referencia"] = df["preco_referencia"].fillna(mediana_geral)

# Índice de posicionamento
df["indice_preco"] = df["Valor FIPE"] / df["preco_referencia"]

# Classes com tolerância de ±10%
condicoes = [
    df["indice_preco"] < 0.90,
    df["indice_preco"].between(0.90, 1.10, inclusive="both"),
    df["indice_preco"] > 1.10
]
classes = [0, 1, 2]

df["posicionamento_preco"] = np.select(condicoes, classes, default=1)

mapa_classes = {
    0: "Abaixo do esperado",
    1: "Dentro do esperado",
    2: "Acima do esperado"
}

df["classe_nome"] = df["posicionamento_preco"].map(mapa_classes)

display(df[[
    "Montadora","Modelo","Ano Modelo","Valor FIPE",
    "preco_referencia","indice_preco","classe_nome"
]].head(10))


### 3.1. Distribuição das classes


In [ ]:
dist = (
    df["classe_nome"]
    .value_counts()
    .reindex(["Abaixo do esperado", "Dentro do esperado", "Acima do esperado"])
)

print("Distribuição absoluta:")
print(dist)

print("\nDistribuição percentual:")
print((dist / len(df) * 100).round(2))

sns.countplot(
    data=df,
    x="classe_nome",
    order=["Abaixo do esperado", "Dentro do esperado", "Acima do esperado"]
)
plt.title("Distribuição das Classes de Posicionamento de Preço")
plt.xlabel("Posicionamento")
plt.ylabel("Quantidade de veículos")
plt.xticks(rotation=10)
plt.show()


## 4. Análise Exploratória

Antes da modelagem, analisamos como o posicionamento de preço se comporta entre anos e montadoras.


In [ ]:
print("Resumo do Valor FIPE:")
display(df["Valor FIPE"].describe())

print("\nTop 10 montadoras:")
display(df["Montadora"].value_counts().head(10))

# Boxplot por classe
sns.boxplot(data=df, x="classe_nome", y="indice_preco",
            order=["Abaixo do esperado","Dentro do esperado","Acima do esperado"])
plt.axhline(0.90, linestyle="--")
plt.axhline(1.10, linestyle="--")
plt.title("Índice de Preço por Classe")
plt.xlabel("Classe")
plt.ylabel("Valor FIPE / Preço de Referência")
plt.show()


In [ ]:
# Perfil médio por classe
perfil_classes = (
    df.groupby("classe_nome")
      .agg(
          qtd=("Valor FIPE","size"),
          valor_fipe_mediano=("Valor FIPE","median"),
          ano_mediano=("Ano Modelo","median"),
          indice_medio=("indice_preco","mean")
      )
      .round(2)
)

display(perfil_classes)


## 5. Preparação para Machine Learning

### Features utilizadas

- Montadora
- Modelo
- Cor
- Grupo
- Situação
- Ano Modelo

### Target

`posicionamento_preco`

O modelo **não recebe**:

- Valor FIPE
- preço de referência
- índice de preço

Essas variáveis revelariam direta ou indiretamente a classe correta.


In [ ]:
features_categoricas = ["Montadora", "Modelo", "Cor", "Grupo", "Situação"]
features_numericas = ["Ano Modelo"]

X = df[features_categoricas + features_numericas].copy()
y = df["posicionamento_preco"].copy()

print("Features:", X.columns.tolist())
print("Target: posicionamento_preco")
print("\nDistribuição do target:")
print(y.value_counts(normalize=True).sort_index().round(3))


### 5.1. Separação treino/teste


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Treino: {len(X_train)} registros")
print(f"Teste: {len(X_test)} registros")


### 5.2. Pipeline de pré-processamento


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_categoricas)
    ]
)

print("Pré-processador criado.")


## 6. Modelagem

Serão comparados três modelos:

1. **Dummy Classifier** — baseline mínimo;
2. **Regressão Logística Multinomial** — baseline supervisionado e interpretável;
3. **Random Forest Classifier** — modelo principal, capaz de capturar relações não lineares.


In [ ]:
modelos = {
    "Dummy Baseline": DummyClassifier(strategy="most_frequent"),
    "Regressão Logística": LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

pipelines = {}
resultados = []

for nome, modelo in modelos.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", modelo)
    ])

    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    pipelines[nome] = pipeline

    resultados.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_test, pred),
        "Precisão Ponderada": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall Ponderado": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1 Ponderado": f1_score(y_test, pred, average="weighted", zero_division=0),
        "F1 Macro": f1_score(y_test, pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(resultados).set_index("Modelo")
display(results_df.round(4))


### Por que usar F1 Macro?

Como as três classes podem apresentar tamanhos diferentes, o F1 Macro dá o mesmo peso a cada classe. Isso evita que um modelo pareça bom apenas por acertar a classe majoritária.


## 7. Otimização do Random Forest


In [ ]:
pipeline_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

param_grid = {
    "classifier__n_estimators": [150, 250],
    "classifier__max_depth": [10, 20, None],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    pipeline_rf,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

rf_otimizado = grid.best_estimator_

print("Melhores parâmetros:")
print(grid.best_params_)
print(f"Melhor F1 Macro na validação cruzada: {grid.best_score_:.4f}")


## 8. Avaliação Final


In [ ]:
pred_otimizado = rf_otimizado.predict(X_test)

metricas_otimizado = pd.DataFrame({
    "Métrica": ["Acurácia","Precisão Ponderada","Recall Ponderado","F1 Ponderado","F1 Macro"],
    "Valor": [
        accuracy_score(y_test, pred_otimizado),
        precision_score(y_test, pred_otimizado, average="weighted", zero_division=0),
        recall_score(y_test, pred_otimizado, average="weighted", zero_division=0),
        f1_score(y_test, pred_otimizado, average="weighted", zero_division=0),
        f1_score(y_test, pred_otimizado, average="macro", zero_division=0)
    ]
})

display(metricas_otimizado.round(4))

print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    pred_otimizado,
    labels=[0,1,2],
    target_names=[
        "Abaixo do esperado",
        "Dentro do esperado",
        "Acima do esperado"
    ],
    zero_division=0
))


### 8.1. Matriz de Confusão


In [ ]:
cm = confusion_matrix(y_test, pred_otimizado, labels=[0,1,2])

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Abaixo","Dentro","Acima"],
    yticklabels=["Abaixo","Dentro","Acima"]
)
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.title("Matriz de Confusão - Random Forest Otimizado")
plt.show()


### 8.2. Overfitting


In [ ]:
acc_treino = rf_otimizado.score(X_train, y_train)
acc_teste = rf_otimizado.score(X_test, y_test)

print(f"Acurácia treino: {acc_treino:.4f}")
print(f"Acurácia teste:  {acc_teste:.4f}")
print(f"Diferença:       {acc_treino - acc_teste:.4f}")

if acc_treino - acc_teste > 0.10:
    print("Possível overfitting: a diferença entre treino e teste é relevante.")
else:
    print("Não há indicação forte de overfitting pelo critério adotado.")


## 9. Importância das Variáveis


In [ ]:
feature_names = (
    rf_otimizado
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

importancias = (
    rf_otimizado
    .named_steps["classifier"]
    .feature_importances_
)

imp_df = (
    pd.DataFrame({"Feature": feature_names, "Importância": importancias})
      .sort_values("Importância", ascending=False)
)

display(imp_df.head(20))

sns.barplot(data=imp_df.head(20), x="Importância", y="Feature")
plt.title("20 Features Mais Importantes")
plt.tight_layout()
plt.show()


## 10. Análise dos Erros


In [ ]:
analise = X_test.copy()
analise["classe_real"] = y_test.values
analise["classe_prevista"] = pred_otimizado
analise["acerto"] = analise["classe_real"] == analise["classe_prevista"]

analise["classe_real_nome"] = analise["classe_real"].map(mapa_classes)
analise["classe_prevista_nome"] = analise["classe_prevista"].map(mapa_classes)

print("Acertos e erros:")
print(analise["acerto"].value_counts())

print("\nExemplos de erros:")
display(
    analise.loc[
        ~analise["acerto"],
        [
            "Montadora","Modelo","Ano Modelo","Grupo","Situação",
            "classe_real_nome","classe_prevista_nome"
        ]
    ].head(15)
)


## 11. Conclusão

Este MVP reformula a base de veículos como um problema de **classificação multiclasse**, em vez de repetir a previsão direta do Valor FIPE.

O projeto busca identificar se um veículo apresenta preço:

- abaixo do padrão de veículos comparáveis;
- dentro de uma faixa considerada esperada;
- acima do padrão.

A abordagem possui uma aplicação prática em análise de portfólio, gestão de frotas e triagem comercial, pois permite destacar rapidamente veículos cujo valor se distancia do benchmark de sua montadora e ano.

### Limitações

A base disponível possui poucas variáveis relacionadas ao estado real, quilometragem, versão, opcionais e localização do veículo. Por isso, o posicionamento de preço deve ser interpretado como uma classificação baseada apenas nas informações disponíveis.

### Próximos passos

- incluir quilometragem, versão, combustível e câmbio;
- criar benchmarks mais granulares quando houver volume suficiente;
- testar Gradient Boosting/XGBoost;
- calibrar a tolerância de ±10%;
- avaliar SHAP para explicabilidade;
- criar uma interface para consulta de novos veículos.
